In [2]:
import numpy as np
import pandas as pd
import torch
import os, json, copy, time
torch.set_printoptions(precision=4, sci_mode=False)

from omegaconf import OmegaConf

import pybullet as p
import fcl

from tqdm import tqdm, trange
from torchmetrics import AUROC
get_auroc = AUROC(task='binary')

import plotly 
import plotly.graph_objs as go
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.io as pio
import matplotlib.pyplot as plt

from envs.models.panda.multipanda_bullet import MultiPanda_bullet
from envs.models.panda.panda import Panda
from envs.multipanda import MultiPanda

from envs.lib.LieGroup import invSE3

from envs import get_env
from training.model import get_model
from training.loader import get_dataloader

from training.model.PairwiseNet import Pairwise2Global
# from utils import get_model_from_result_dir

device = 'cuda:0'

pybullet build time: May 20 2022 19:45:31


Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [4]:
env_config_paths = [
    'configs/envs/env_config_2arm.yml',
    'configs/envs/env_config_3arm.yml',
    'configs/envs/env_config_4arm.yml',
]

df_results = pd.DataFrame(columns=['env', 'method', 'time', 'mse'])

for env_config_path in env_config_paths:
    env_cfg = OmegaConf.load(env_config_path)
    env = get_env(env_cfg, mesh_type='original')

    N = 1000
    q = torch.rand(N, env.n_dof) * (env.q_max - env.q_min).unsqueeze(0).repeat(N, 1) + env.q_min.unsqueeze(0).repeat(N, 1)

    collision_pairs = []
    for o1_idx in range(env.n_objects):
        for o2_idx in range(o1_idx+1, env.n_objects):
            b1ID, l1ID = env.env_bullet.idx2id(o1_idx)
            b2ID, l2ID = env.env_bullet.idx2id(o2_idx)
            if b1ID == b2ID:
                continue
            collision_pairs.append([o1_idx, o2_idx])
            
    print(f'{env.id} : {len(collision_pairs)} pairs')
    
    time_elapsed = 0

    d_original = torch.zeros(N, 1)

    for idx in range(N):
        
        jointPos = q[idx]
        
        fcl_mgrs = []
        start_idx = 0
        for r_idx in range(env.n_robot):
            tmp_mgr = fcl.DynamicAABBTreeCollisionManager()
            tmp_mgr.registerObjects(env.robots[r_idx].fcl_objs(jointPos[start_idx:start_idx+env.robots[r_idx].n_dof]))
            fcl_mgrs.append(tmp_mgr)
            start_idx += env.robots[r_idx].n_dof
            
        t_start = time.time()

        min_distance = 1e10
        for i_idx in range(env.n_robot):
            for j_idx in range(i_idx+1, env.n_robot):
                tmp_mgr1 = fcl_mgrs[i_idx]
                tmp_mgr2 = fcl_mgrs[j_idx]

                tmp_mgr1.setup()
                tmp_mgr2.setup()

                req = fcl.CollisionRequest(enable_contact=True, enable_cost=True, gjk_solver_type=fcl.GJKSolverType.GST_LIBCCD)
                rdata = fcl.CollisionData(request = req)
                tmp_mgr1.collide(tmp_mgr2, rdata, fcl.defaultCollisionCallback)

                isCollision = rdata.result.is_collision

                if isCollision:
                    max_pd_depth = -1e10
                    for contact in rdata.result.contacts:
                        if contact.penetration_depth > max_pd_depth:
                            max_pd_depth = contact.penetration_depth

                    distance = -max_pd_depth

                else:
                    req = fcl.DistanceRequest(enable_nearest_points=True, gjk_solver_type=fcl.GJKSolverType.GST_LIBCCD)
                    ddata = fcl.DistanceData(request = req)
                    tmp_mgr1.distance(tmp_mgr2, ddata, fcl.defaultDistanceCallback)

                    distance = ddata.result.min_distance

                if min_distance > distance:
                    min_distance = distance
                    
        d_original[idx] = min_distance
        
        time_elapsed += time.time() - t_start

    print(f'FCL (original) : {time_elapsed:.4f} sec')
    result_original = {'env': env.id, 'method': 'original', 'time': time_elapsed, 'mse': None}
    df_results = df_results.append(pd.DataFrame([result_original]), ignore_index=True)
    
    env = get_env(env_cfg, mesh_type='simplified')
    time_elapsed = 0
    d_convex = torch.zeros(N, 1)

    for idx, jointPos in enumerate(q):
        
        fcl_mgrs = []
        start_idx = 0
        for r_idx in range(env.n_robot):
            tmp_mgr = fcl.DynamicAABBTreeCollisionManager()
            tmp_mgr.registerObjects(env.robots[r_idx].convex_fcl_objs(jointPos[start_idx:start_idx+env.robots[r_idx].n_dof]))
            fcl_mgrs.append(tmp_mgr)
            start_idx += env.robots[r_idx].n_dof
            
        t_start = time.time()

        min_distance = 1e10
        for i_idx in range(env.n_robot):
            for j_idx in range(i_idx+1, env.n_robot):
                tmp_mgr1 = fcl_mgrs[i_idx]
                tmp_mgr2 = fcl_mgrs[j_idx]

                tmp_mgr1.setup()
                tmp_mgr2.setup()

                req = fcl.CollisionRequest(enable_contact=True, enable_cost=True, gjk_solver_type=fcl.GJKSolverType.GST_LIBCCD)
                rdata = fcl.CollisionData(request = req)
                tmp_mgr1.collide(tmp_mgr2, rdata, fcl.defaultCollisionCallback)

                isCollision = rdata.result.is_collision

                if isCollision:
                    max_pd_depth = -1e10
                    for contact in rdata.result.contacts:
                        if contact.penetration_depth > max_pd_depth:
                            max_pd_depth = contact.penetration_depth

                    distance = -max_pd_depth

                else:
                    req = fcl.DistanceRequest(enable_nearest_points=True, gjk_solver_type=fcl.GJKSolverType.GST_LIBCCD)
                    ddata = fcl.DistanceData(request = req)
                    tmp_mgr1.distance(tmp_mgr2, ddata, fcl.defaultDistanceCallback)

                    distance = ddata.result.min_distance

                if min_distance > distance:
                    min_distance = distance
                    
        d_convex[idx] = min_distance
        
        time_elapsed += time.time() - t_start

    print(f'FCL (convex) : {time_elapsed:.4f} sec')
    result_convex = {'env': env.id, 'method': 'convex', 'time': time_elapsed, 'mse': torch.nn.MSELoss()(d_original, d_convex).item()}
    df_results = df_results.append(pd.DataFrame([result_convex]), ignore_index=True)
    
    time_elapsed = 0

    d_capsule = torch.zeros(N, 1)

    for idx, jointPos in enumerate(q):

        ################################# Global Collision Check 
        fcl_mgrs = []
        start_idx = 0
        for r_idx in range(env.n_robot):
            tmp_mgr = fcl.DynamicAABBTreeCollisionManager()
            tmp_mgr.registerObjects(env.robots[r_idx].capsule_fcl_objs(jointPos[start_idx:start_idx+env.robots[r_idx].n_dof]))
            fcl_mgrs.append(tmp_mgr)
            start_idx += env.robots[r_idx].n_dof
            
        t_start = time.time()

        min_distance = 1e10
        for i_idx in range(env.n_robot):
            for j_idx in range(i_idx+1, env.n_robot):
                tmp_mgr1 = fcl_mgrs[i_idx]
                tmp_mgr2 = fcl_mgrs[j_idx]

                tmp_mgr1.setup()
                tmp_mgr2.setup()

                req = fcl.CollisionRequest(enable_contact=True, enable_cost=True, gjk_solver_type=fcl.GJKSolverType.GST_LIBCCD)
                rdata = fcl.CollisionData(request = req)
                tmp_mgr1.collide(tmp_mgr2, rdata, fcl.defaultCollisionCallback)

                isCollision = rdata.result.is_collision

                if isCollision:
                    max_pd_depth = -1e10
                    for contact in rdata.result.contacts:
                        if contact.penetration_depth > max_pd_depth:
                            max_pd_depth = contact.penetration_depth

                    distance = -max_pd_depth

                else:
                    req = fcl.DistanceRequest(enable_nearest_points=True, gjk_solver_type=fcl.GJKSolverType.GST_LIBCCD)
                    ddata = fcl.DistanceData(request = req)
                    tmp_mgr1.distance(tmp_mgr2, ddata, fcl.defaultDistanceCallback)

                    distance = ddata.result.min_distance

                if min_distance > distance:
                    min_distance = distance
                    
        d_capsule[idx] = min_distance
        
        time_elapsed += time.time() - t_start

    print(f'FCL (capsule) : {time_elapsed:.4f} sec')
    result_capsule = {'env': env.id, 'method': 'capsule', 'time': time_elapsed, 'mse': torch.nn.MSELoss()(d_original, d_capsule).item()}
    df_results = df_results.append(pd.DataFrame([result_capsule]), ignore_index=True)

/PublicSSD/jihwankim/PairwiseNet/envs/models/panda/panda.py:220: UserWarning: An output with one or more elements was resized since it had shape [1, 3, 3], which does not match the required output shape [1, 1, 3, 3]. This behavior is deprecated, and in a future PyTorch release outputs will not be resized unless they have zero elements. You can explicitly reuse an out tensor t by resizing it, inplace, to zero elements with t.resize_(0). (Triggered internally at  /opt/conda/conda-bld/pytorch_1659484809662/work/aten/src/ATen/native/Resize.cpp:17.)
  T_link[:3, :3] = torch.matrix_exp(skew_so3((w*theta).unsqueeze(0)).squeeze(0))


twoarm : 64 pairs
FCL (original) : 11.4807 sec


/tmp/ipykernel_3121851/1651893051.py:84: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df_results = df_results.append(pd.DataFrame([result_original]), ignore_index=True)
/PublicSSD/jihwankim/PairwiseNet/envs/models/panda/panda.py:220: UserWarning: An output with one or more elements was resized since it had shape [1, 3, 3], which does not match the required output shape [1, 1, 3, 3]. This behavior is deprecated, and in a future PyTorch release outputs will not be resized unless they have zero elements. You can explicitly reuse an out tensor t by resizing it, inplace, to zero elements with t.resize_(0). (Triggered internally at  /opt/conda/conda-bld/pytorch_1659484809662/work/aten/src/ATen/native/Resize.cpp:17.)
  T_link[:3, :3] = torch.matrix_exp(skew_so3((w*theta).unsqueeze(0)).squeeze(0))


FCL (convex) : 0.0714 sec


/tmp/ipykernel_3121851/1651893051.py:141: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df_results = df_results.append(pd.DataFrame([result_convex]), ignore_index=True)


...

/tmp/ipykernel_3121851/1651893051.py:84: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df_results = df_results.append(pd.DataFrame([result_original]), ignore_index=True)
/PublicSSD/jihwankim/PairwiseNet/envs/models/panda/panda.py:220: UserWarning: An output with one or more elements was resized since it had shape [1, 3, 3], which does not match the required output shape [1, 1, 3, 3]. This behavior is deprecated, and in a future PyTorch release outputs will not be resized unless they have zero elements. You can explicitly reuse an out tensor t by resizing it, inplace, to zero elements with t.resize_(0). (Triggered internally at  /opt/conda/conda-bld/pytorch_1659484809662/work/aten/src/ATen/native/Resize.cpp:17.)
  T_link[:3, :3] = torch.matrix_exp(skew_so3((w*theta).unsqueeze(0)).squeeze(0))


FCL (convex) : 0.3265 sec


/tmp/ipykernel_3121851/1651893051.py:141: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df_results = df_results.append(pd.DataFrame([result_convex]), ignore_index=True)


FCL (capsule) : 0.1130 sec


/tmp/ipykernel_3121851/1651893051.py:199: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df_results = df_results.append(pd.DataFrame([result_capsule]), ignore_index=True)


In [6]:
df_results.to_csv('results/fcl_comparison.csv', index=False)

In [13]:
pd.read_csv('results/fcl_comparison.csv')

,env,method,time,mse
0,twoarm,original,11.480741,NaN
1,twoarm,convex,0.071387,0.000455
2,twoarm,capsule,0.029357,0.001376
3,threearm,original,43.301925,NaN
4,threearm,convex,0.168188,0.000227
5,threearm,capsule,0.058240,0.001048
6,fourarm,original,88.383127,NaN
7,fourarm,convex,0.326525,0.001610
8,fourarm,capsule,0.113046,0.003378


In [4]:
################################# Original Mesh #################################

time_elapsed = 0

d_original = torch.zeros(N, 1)

for idx in trange(N):
    
    jointPos = q[idx]
    
    fcl_mgrs = []
    start_idx = 0
    for r_idx in range(env.n_robot):
        tmp_mgr = fcl.DynamicAABBTreeCollisionManager()
        tmp_mgr.registerObjects(env.robots[r_idx].fcl_objs(jointPos[start_idx:start_idx+env.robots[r_idx].n_dof]))
        fcl_mgrs.append(tmp_mgr)
        start_idx += env.robots[r_idx].n_dof
        
    t_start = time.time()

    min_distance = 1e10
    for i_idx in range(env.n_robot):
        for j_idx in range(i_idx+1, env.n_robot):
            tmp_mgr1 = fcl_mgrs[i_idx]
            tmp_mgr2 = fcl_mgrs[j_idx]

            tmp_mgr1.setup()
            tmp_mgr2.setup()

            req = fcl.CollisionRequest(enable_contact=True, enable_cost=True, gjk_solver_type=fcl.GJKSolverType.GST_LIBCCD)
            rdata = fcl.CollisionData(request = req)
            tmp_mgr1.collide(tmp_mgr2, rdata, fcl.defaultCollisionCallback)

            isCollision = rdata.result.is_collision

            if isCollision:
                max_pd_depth = -1e10
                for contact in rdata.result.contacts:
                    if contact.penetration_depth > max_pd_depth:
                        max_pd_depth = contact.penetration_depth

                distance = -max_pd_depth

            else:
                req = fcl.DistanceRequest(enable_nearest_points=True, gjk_solver_type=fcl.GJKSolverType.GST_LIBCCD)
                ddata = fcl.DistanceData(request = req)
                tmp_mgr1.distance(tmp_mgr2, ddata, fcl.defaultDistanceCallback)

                distance = ddata.result.min_distance

            if min_distance > distance:
                min_distance = distance
                
    d_original[idx] = min_distance
    
    time_elapsed += time.time() - t_start

print(f'FCL (original) : {time_elapsed:.4f} sec')

100%|██████████| 1000/1000 [01:35<00:00, 10.48it/s]

FCL (original) : 88.9803 sec


In [5]:
env_cfg = OmegaConf.load('configs/envs/env_config_4arm.yml')
env = get_env(env_cfg, mesh_type='simplified')

In [6]:
time_elapsed = 0

d_convex = torch.zeros(N, 1)

for idx, jointPos in enumerate(q):
    
    # ################################# Pairwise Collision Check (No broad narrow phase)
    # start_idx = 0
    # convex_fcl_objs = []
    # for r_idx in range(env.n_robot):
    #     convex_fcl_objs += env.robots[r_idx].fcl_objs(jointPos[start_idx:start_idx+env.robots[r_idx].n_dof])
    # env.set_inputs(jointPos)
    
    # t_start = time.time()
    
    # min_distance = 1e10
    # for pair in collision_pairs:
    #     tmp_mgr1 = fcl.DynamicAABBTreeCollisionManager()
    #     tmp_mgr2 = fcl.DynamicAABBTreeCollisionManager()
        
    #     tmp_mgr1.registerObject(convex_fcl_objs[pair[0]])
    #     tmp_mgr2.registerObject(convex_fcl_objs[pair[1]])
        
    #     tmp_mgr1.setup()
    #     tmp_mgr2.setup()
        
    #     req = fcl.CollisionRequest(enable_contact=True, enable_cost=True, gjk_solver_type=fcl.GJKSolverType.GST_LIBCCD)
    #     rdata = fcl.CollisionData(request = req)
    #     tmp_mgr1.collide(tmp_mgr2, rdata, fcl.defaultCollisionCallback)

    #     isCollision = rdata.result.is_collision

    #     if isCollision:
    #         max_pd_depth = -1e10
    #         for contact in rdata.result.contacts:
    #             if contact.penetration_depth > max_pd_depth:
    #                 max_pd_depth = contact.penetration_depth

    #         distance = -max_pd_depth

    #     else:
    #         req = fcl.DistanceRequest(enable_nearest_points=True, gjk_solver_type=fcl.GJKSolverType.GST_LIBCCD)
    #         ddata = fcl.DistanceData(request = req)
    #         tmp_mgr1.distance(tmp_mgr2, ddata, fcl.defaultDistanceCallback)

    #         distance = ddata.result.min_distance

    #     if min_distance > distance:
    #         min_distance = distance
            
    # time_elapsed += time.time() - t_start
            
    ################################# Global Collision Check 
    fcl_mgrs = []
    start_idx = 0
    for r_idx in range(env.n_robot):
        tmp_mgr = fcl.DynamicAABBTreeCollisionManager()
        tmp_mgr.registerObjects(env.robots[r_idx].convex_fcl_objs(jointPos[start_idx:start_idx+env.robots[r_idx].n_dof]))
        fcl_mgrs.append(tmp_mgr)
        start_idx += env.robots[r_idx].n_dof
        
    t_start = time.time()

    min_distance = 1e10
    for i_idx in range(env.n_robot):
        for j_idx in range(i_idx+1, env.n_robot):
            tmp_mgr1 = fcl_mgrs[i_idx]
            tmp_mgr2 = fcl_mgrs[j_idx]

            tmp_mgr1.setup()
            tmp_mgr2.setup()

            req = fcl.CollisionRequest(enable_contact=True, enable_cost=True, gjk_solver_type=fcl.GJKSolverType.GST_LIBCCD)
            rdata = fcl.CollisionData(request = req)
            tmp_mgr1.collide(tmp_mgr2, rdata, fcl.defaultCollisionCallback)

            isCollision = rdata.result.is_collision

            if isCollision:
                max_pd_depth = -1e10
                for contact in rdata.result.contacts:
                    if contact.penetration_depth > max_pd_depth:
                        max_pd_depth = contact.penetration_depth

                distance = -max_pd_depth

            else:
                req = fcl.DistanceRequest(enable_nearest_points=True, gjk_solver_type=fcl.GJKSolverType.GST_LIBCCD)
                ddata = fcl.DistanceData(request = req)
                tmp_mgr1.distance(tmp_mgr2, ddata, fcl.defaultDistanceCallback)

                distance = ddata.result.min_distance

            if min_distance > distance:
                min_distance = distance
                
    d_convex[idx] = min_distance
    
    time_elapsed += time.time() - t_start

print(f'FCL (convex) : {time_elapsed:.4f} sec')

FCL (convex) : 0.3301 sec


In [7]:
torch.nn.MSELoss()(d_convex, d_original).item()

0.0015225281240418553

In [8]:
time_elapsed = 0

d_capsule = torch.zeros(N, 1)

for idx, jointPos in enumerate(q):
    
    # ################################# Pairwise Collision Check (No broad narrow phase)
    # start_idx = 0
    # convex_fcl_objs = []
    # for r_idx in range(env.n_robot):
    #     convex_fcl_objs += env.robots[r_idx].fcl_objs(jointPos[start_idx:start_idx+env.robots[r_idx].n_dof])
    # env.set_inputs(jointPos)
    
    # t_start = time.time()
    
    # min_distance = 1e10
    # for pair in collision_pairs:
    #     tmp_mgr1 = fcl.DynamicAABBTreeCollisionManager()
    #     tmp_mgr2 = fcl.DynamicAABBTreeCollisionManager()
        
    #     tmp_mgr1.registerObject(convex_fcl_objs[pair[0]])
    #     tmp_mgr2.registerObject(convex_fcl_objs[pair[1]])
        
    #     tmp_mgr1.setup()
    #     tmp_mgr2.setup()
        
    #     req = fcl.CollisionRequest(enable_contact=True, enable_cost=True, gjk_solver_type=fcl.GJKSolverType.GST_LIBCCD)
    #     rdata = fcl.CollisionData(request = req)
    #     tmp_mgr1.collide(tmp_mgr2, rdata, fcl.defaultCollisionCallback)

    #     isCollision = rdata.result.is_collision

    #     if isCollision:
    #         max_pd_depth = -1e10
    #         for contact in rdata.result.contacts:
    #             if contact.penetration_depth > max_pd_depth:
    #                 max_pd_depth = contact.penetration_depth

    #         distance = -max_pd_depth

    #     else:
    #         req = fcl.DistanceRequest(enable_nearest_points=True, gjk_solver_type=fcl.GJKSolverType.GST_LIBCCD)
    #         ddata = fcl.DistanceData(request = req)
    #         tmp_mgr1.distance(tmp_mgr2, ddata, fcl.defaultDistanceCallback)

    #         distance = ddata.result.min_distance

    #     if min_distance > distance:
    #         min_distance = distance
            
    # time_elapsed += time.time() - t_start
            
    ################################# Global Collision Check 
    fcl_mgrs = []
    start_idx = 0
    for r_idx in range(env.n_robot):
        tmp_mgr = fcl.DynamicAABBTreeCollisionManager()
        tmp_mgr.registerObjects(env.robots[r_idx].capsule_fcl_objs(jointPos[start_idx:start_idx+env.robots[r_idx].n_dof]))
        fcl_mgrs.append(tmp_mgr)
        start_idx += env.robots[r_idx].n_dof
        
    t_start = time.time()

    min_distance = 1e10
    for i_idx in range(env.n_robot):
        for j_idx in range(i_idx+1, env.n_robot):
            tmp_mgr1 = fcl_mgrs[i_idx]
            tmp_mgr2 = fcl_mgrs[j_idx]

            tmp_mgr1.setup()
            tmp_mgr2.setup()

            req = fcl.CollisionRequest(enable_contact=True, enable_cost=True, gjk_solver_type=fcl.GJKSolverType.GST_LIBCCD)
            rdata = fcl.CollisionData(request = req)
            tmp_mgr1.collide(tmp_mgr2, rdata, fcl.defaultCollisionCallback)

            isCollision = rdata.result.is_collision

            if isCollision:
                max_pd_depth = -1e10
                for contact in rdata.result.contacts:
                    if contact.penetration_depth > max_pd_depth:
                        max_pd_depth = contact.penetration_depth

                distance = -max_pd_depth

            else:
                req = fcl.DistanceRequest(enable_nearest_points=True, gjk_solver_type=fcl.GJKSolverType.GST_LIBCCD)
                ddata = fcl.DistanceData(request = req)
                tmp_mgr1.distance(tmp_mgr2, ddata, fcl.defaultDistanceCallback)

                distance = ddata.result.min_distance

            if min_distance > distance:
                min_distance = distance
                
    d_capsule[idx] = min_distance
    
    time_elapsed += time.time() - t_start

print(f'FCL (capsule) : {time_elapsed:.4f} sec')

FCL (capsule) : 0.1108 sec


In [9]:
torch.nn.MSELoss()(d_capsule, d_original).item()

0.003198486752808094

In [48]:
import torch
import torch.nn as nn
import time

class ConcatNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.ModuleList([
            nn.Linear(76, 128),    # 첫 번째 층 (76 = 32 + 32 + 12)
            nn.Linear(128, 128),   # 두 번째 층
            nn.Linear(128, 128),   # 세 번째 층
            nn.Linear(128, 1)      # 출력 층
        ])
    
    def forward(self, x1, x2, x3):
        # 입력 concatenate
        x = torch.cat([x1, x2, x3], dim=1)
        
        # 각 층의 입력과 활성화값 저장
        self.pre_activations = []
        self.activations = []
        
        for i, layer in enumerate(self.layers):
            # linear 변환
            z = layer(x)
            self.pre_activations.append(z)
            
            # 마지막 층이 아니면 sigmoid 적용
            if i < len(self.layers) - 1:
                x = torch.sigmoid(z)
                self.activations.append(x)
            else:
                x = z  # 마지막 층은 활성화 함수 없음
        
        return x
    
    def compute_analytic_jacobian_x3(self, x1, x2, x3):
        """x3(12차원)에 대한 Jacobian을 batch 연산으로 계산"""
        batch_size = x3.shape[0]
        
        # Forward pass
        _ = self.forward(x1, x2, x3)
        
        # 각 층의 가중치
        W1, W2, W3, W4 = [layer.weight for layer in self.layers]
        
        # Sigmoid 미분값 계산
        def sigmoid_derivative(x):
            sigmoid_x = torch.sigmoid(x)
            return sigmoid_x * (1 - sigmoid_x)
        
        # 각 층의 sigmoid 미분값 (batch, dim) 형태
        d1 = sigmoid_derivative(self.pre_activations[0])  # (batch, 128)
        d2 = sigmoid_derivative(self.pre_activations[1])  # (batch, 128)
        d3 = sigmoid_derivative(self.pre_activations[2])  # (batch, 128)
        
        # x3에 해당하는 W1의 부분만 추출 (마지막 12열)
        W1_x3 = W1[:, -12:]  # (128, 12)
        
        # 배치 전체에 대한 역전파 계산
        # W4: (1, 128)
        # d3: (batch, 128)
        # W3: (128, 128)
        # d2: (batch, 128)
        # W2: (128, 128)
        # d1: (batch, 128)
        # W1_x3: (128, 12)
        
        # 각 배치 요소에 대해 요소별 곱을 행렬 연산으로 처리
        grad = W4.expand(batch_size, -1)  # (batch, 128)
        grad = grad * d3  # (batch, 128)
        grad = grad @ W3  # (batch, 128)
        grad = grad * d2  # (batch, 128)
        grad = grad @ W2  # (batch, 128)
        grad = grad * d1  # (batch, 128)
        grad = grad @ W1_x3  # (batch, 12)
        
        # 최종 형태로 변환 (batch, 1, 12)
        return grad.unsqueeze(1)

# 테스트 및 벤치마크 코드
def test_and_benchmark(n_pair=64, batch=False, device='cpu'):
    # 모델 및 데이터 생성
    if batch:
        batch_size = n_pair * 1000
    else:
        batch_size = n_pair
        
    model = ConcatNetwork().to(device)
    x1 = torch.randn(batch_size, 32).to(device)
    x2 = torch.randn(batch_size, 32).to(device)
    x3 = torch.randn(batch_size, 12).to(device)
    
    # Analytic Jacobian 계산 시간 측정
    start_time = time.time()
    if batch:
        jacobian = model.compute_analytic_jacobian_x3(x1, x2, x3)
    else:
        for _ in range(1000):
            jacobian = model.compute_analytic_jacobian_x3(x1, x2, x3)
    analytic_time = time.time() - start_time

    print(f"N_pair: {n_pair}, Batch: {batch}, Device: {device}")
    print(f"Time: {analytic_time:.4f} sec")

if __name__ == "__main__":
    test_and_benchmark(64,  batch=False, device='cpu')
    test_and_benchmark(64,  batch=False, device='cuda:0')
    test_and_benchmark(192, batch=False, device='cpu')
    test_and_benchmark(192, batch=False, device='cuda:0')
    test_and_benchmark(384, batch=False, device='cpu')
    test_and_benchmark(384, batch=False, device='cuda:0')
    
    test_and_benchmark(64,  batch=True, device='cpu')
    test_and_benchmark(64,  batch=True, device='cuda:0')
    test_and_benchmark(192, batch=True, device='cpu')
    test_and_benchmark(192, batch=True, device='cuda:0')
    test_and_benchmark(384, batch=True, device='cpu')
    test_and_benchmark(384, batch=True, device='cuda:0')

N_pair: 64, Batch: False, Device: cpu
Time: 0.1813 sec
N_pair: 64, Batch: False, Device: cuda:0
Time: 0.1484 sec
N_pair: 192, Batch: False, Device: cpu
Time: 0.3038 sec
N_pair: 192, Batch: False, Device: cuda:0
Time: 0.1380 sec
N_pair: 384, Batch: False, Device: cpu
Time: 0.3974 sec
N_pair: 384, Batch: False, Device: cuda:0
Time: 0.1407 sec
N_pair: 64, Batch: True, Device: cpu
Time: 0.0697 sec
N_pair: 64, Batch: True, Device: cuda:0
Time: 0.0004 sec
N_pair: 192, Batch: True, Device: cpu
Time: 0.2257 sec
N_pair: 192, Batch: True, Device: cuda:0
Time: 0.0004 sec
N_pair: 384, Batch: True, Device: cpu
Time: 0.4727 sec
N_pair: 384, Batch: True, Device: cuda:0
Time: 0.0004 sec


In [50]:
cfg = OmegaConf.load('configs/training/config_PairwiseNet_4arm.yml')
device = 'cuda:0'
model = get_model(cfg.model)
for device in ['cpu', 'cuda:0']:
    model = model.to(device)
    tmp1 = torch.randn(pair, 32).to(device)
    tmp2 = torch.randn(pair, 32).to(device)
    tmp3 = torch.randn(pair, 12).to(device)
    tmp = model.forward_from_embed(tmp1, tmp2, tmp3)
    
    for pair in [64, 192, 384]:
        tmp1 = torch.randn(pair, 32).to(device)
        tmp2 = torch.randn(pair, 32).to(device)
        tmp3 = torch.randn(pair, 12).to(device)
        
        start_time = time.time()
        for _ in range(1000):
            tmp = model.forward_from_embed(tmp1, tmp2, tmp3)
            tmp = tmp.min(dim=0).values
        print(f"device : {device}, pair : {pair}, batch : False, time : {time.time() - start_time:.5f} seconds")
        
        tmp1 = torch.randn(pair*1000, 32).to(device)
        tmp2 = torch.randn(pair*1000, 32).to(device)
        tmp3 = torch.randn(pair*1000, 12).to(device)

        start_time = time.time()
        tmp = model.forward_from_embed(tmp1, tmp2, tmp3)
        tmp = tmp.view(1000, pair, 1).min(dim=1).values
        print(f"device : {device:}, pair : {pair}, batch : True, time : {time.time() - start_time:.5f} seconds")

device : cpu, pair : 64, batch : False, time : 0.07729 seconds
device : cpu, pair : 64, batch : True, time : 0.02462 seconds
device : cpu, pair : 192, batch : False, time : 0.14952 seconds
device : cpu, pair : 192, batch : True, time : 0.10688 seconds
device : cpu, pair : 384, batch : False, time : 0.20218 seconds
device : cpu, pair : 384, batch : True, time : 0.22048 seconds
device : cuda:0, pair : 64, batch : False, time : 0.06299 seconds
device : cuda:0, pair : 64, batch : True, time : 0.00014 seconds
device : cuda:0, pair : 192, batch : False, time : 0.06389 seconds
device : cuda:0, pair : 192, batch : True, time : 0.00021 seconds
device : cuda:0, pair : 384, batch : False, time : 0.06525 seconds
device : cuda:0, pair : 384, batch : True, time : 0.00030 seconds


In [10]:
robot = Panda(mesh_type='original')

/PublicSSD/jihwankim/PairwiseNet/envs/models/panda/panda.py:220: UserWarning: An output with one or more elements was resized since it had shape [1, 3, 3], which does not match the required output shape [1, 1, 3, 3]. This behavior is deprecated, and in a future PyTorch release outputs will not be resized unless they have zero elements. You can explicitly reuse an out tensor t by resizing it, inplace, to zero elements with t.resize_(0). (Triggered internally at  /opt/conda/conda-bld/pytorch_1659484809662/work/aten/src/ATen/native/Resize.cpp:17.)
  T_link[:3, :3] = torch.matrix_exp(skew_so3((w*theta).unsqueeze(0)).squeeze())


In [11]:
for v, t in zip(robot.vertices, robot.triangles):
    print(v.shape, t.shape)

torch.Size([59517, 3]) torch.Size([20483, 3])
torch.Size([37309, 3]) torch.Size([12516, 3])
torch.Size([37871, 3]) torch.Size([12716, 3])
torch.Size([42510, 3]) torch.Size([14233, 3])
torch.Size([43506, 3]) torch.Size([14621, 3])
torch.Size([54751, 3]) torch.Size([18327, 3])
torch.Size([63895, 3]) torch.Size([21620, 3])
torch.Size([35768, 3]) torch.Size([12082, 3])


In [12]:
robot = Panda(mesh_type='simplified')
for v, t in zip(robot.convex_vertices, robot.convex_triangles):
    print(v.shape, t.shape)

torch.Size([102, 3]) torch.Size([200, 3])
torch.Size([152, 3]) torch.Size([300, 3])
torch.Size([152, 3]) torch.Size([300, 3])
torch.Size([152, 3]) torch.Size([300, 3])
torch.Size([152, 3]) torch.Size([300, 3])
torch.Size([152, 3]) torch.Size([300, 3])
torch.Size([102, 3]) torch.Size([200, 3])
torch.Size([102, 3]) torch.Size([200, 3])


/PublicSSD/jihwankim/PairwiseNet/envs/models/panda/panda.py:220: UserWarning: An output with one or more elements was resized since it had shape [1, 3, 3], which does not match the required output shape [1, 1, 3, 3]. This behavior is deprecated, and in a future PyTorch release outputs will not be resized unless they have zero elements. You can explicitly reuse an out tensor t by resizing it, inplace, to zero elements with t.resize_(0). (Triggered internally at  /opt/conda/conda-bld/pytorch_1659484809662/work/aten/src/ATen/native/Resize.cpp:17.)
  T_link[:3, :3] = torch.matrix_exp(skew_so3((w*theta).unsqueeze(0)).squeeze())
